# 📣 Marketing Analytics: Campaign Performance & Attribution Analysis

**Author:** Business Analytics Portfolio Project  
**Tools:** Python, Pandas, Plotly, Scikit-learn, Statsmodels  
**Domain:** Marketing Analytics

---

## 📌 Project Overview

A comprehensive marketing analytics project covering:

1. **Campaign EDA** — Performance metrics across channels and campaigns
2. **Funnel Analysis** — Conversion funnel visualization
3. **A/B Test Analysis** — Statistical significance testing
4. **Marketing Attribution** — First-touch, Last-touch & Linear models
5. **ROAS & ROI Analysis** — Channel profitability breakdown
6. **Budget Optimization** — Recommendations for budget reallocation

---

## 1️⃣ Setup

In [ ]:
!pip install plotly statsmodels --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
pd.set_option('display.float_format', '{:.2f}'.format)
print('✅ Libraries loaded!')

## 2️⃣ Data Generation (Simulated Campaign Data)

In [ ]:
np.random.seed(42)

def generate_campaign_data():
    """Generate realistic digital marketing campaign data."""
    
    channels = ['Google Ads', 'Facebook', 'Instagram', 'Email', 'Organic SEO', 'YouTube']
    campaigns = {
        'Google Ads':    ['Brand Search', 'Non-Brand Search', 'Display Retargeting'],
        'Facebook':      ['Awareness', 'Conversion', 'Lookalike'],
        'Instagram':     ['Story Ads', 'Reel Ads', 'Feed Ads'],
        'Email':         ['Welcome Series', 'Promo Blast', 'Abandoned Cart'],
        'Organic SEO':   ['Blog Traffic', 'Product Pages'],
        'YouTube':       ['Pre-roll Ads', 'Bumper Ads']
    }
    
    channel_params = {
        'Google Ads':  {'cpc': (1.5, 4.5), 'ctr': (0.03, 0.08), 'cvr': (0.04, 0.08), 'spend_share': 0.30},
        'Facebook':    {'cpc': (0.8, 2.5), 'ctr': (0.015,0.04), 'cvr': (0.02, 0.05), 'spend_share': 0.25},
        'Instagram':   {'cpc': (1.0, 3.0), 'ctr': (0.02, 0.05), 'cvr': (0.02, 0.04), 'spend_share': 0.15},
        'Email':       {'cpc': (0.1, 0.3), 'ctr': (0.10, 0.25), 'cvr': (0.05, 0.12), 'spend_share': 0.05},
        'Organic SEO': {'cpc': (0.0, 0.0), 'ctr': (0.20, 0.35), 'cvr': (0.03, 0.06), 'spend_share': 0.05},
        'YouTube':     {'cpc': (0.5, 1.5), 'ctr': (0.01, 0.03), 'cvr': (0.01, 0.03), 'spend_share': 0.20}
    }

    total_monthly_budget = 500000  # ₹5 Lakhs/month
    records = []
    dates = pd.date_range('2024-01-01', '2024-12-31', freq='W')

    for date in dates:
        for channel in channels:
            for campaign in campaigns[channel]:
                p = channel_params[channel]
                spend = (total_monthly_budget * p['spend_share'] / len(campaigns[channel]) / 4.3
                         * np.random.uniform(0.85, 1.15))
                cpc   = np.random.uniform(*p['cpc']) if p['cpc'][1] > 0 else 0.01
                clicks = int(spend / cpc) if cpc > 0 else int(np.random.uniform(500, 2000))
                impressions = int(clicks / np.random.uniform(*p['ctr']))
                conversions = int(clicks * np.random.uniform(*p['cvr']))
                revenue = conversions * np.random.uniform(400, 1500)

                records.append({
                    'Date': date,
                    'Channel': channel,
                    'Campaign': campaign,
                    'Impressions': impressions,
                    'Clicks': clicks,
                    'Spend': round(spend, 2),
                    'Conversions': conversions,
                    'Revenue': round(revenue, 2)
                })

    df = pd.DataFrame(records)
    df['CTR']  = df['Clicks'] / df['Impressions']
    df['CVR']  = df['Conversions'] / df['Clicks'].replace(0, np.nan)
    df['CPC']  = df['Spend'] / df['Clicks'].replace(0, np.nan)
    df['CPA']  = df['Spend'] / df['Conversions'].replace(0, np.nan)
    df['ROAS'] = df['Revenue'] / df['Spend'].replace(0, np.nan)
    df['ROI']  = (df['Revenue'] - df['Spend']) / df['Spend'].replace(0, np.nan)
    return df

df = generate_campaign_data()
print(f'📦 Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'📅 Period: {df.Date.min().date()} → {df.Date.max().date()}')
df.head()

## 3️⃣ Campaign Performance EDA

In [ ]:
# --- Channel-level Summary ---
channel_summary = df.groupby('Channel').agg(
    Total_Spend      = ('Spend', 'sum'),
    Total_Revenue    = ('Revenue', 'sum'),
    Total_Clicks     = ('Clicks', 'sum'),
    Total_Conversions= ('Conversions', 'sum'),
    Avg_CTR          = ('CTR', 'mean'),
    Avg_CVR          = ('CVR', 'mean'),
    Avg_CPA          = ('CPA', 'mean'),
    Avg_ROAS         = ('ROAS', 'mean'),
    Avg_ROI          = ('ROI', 'mean')
).round(3).reset_index()

channel_summary['Profit'] = channel_summary['Total_Revenue'] - channel_summary['Total_Spend']
print('📊 Channel Performance Summary:')
channel_summary.sort_values('Avg_ROAS', ascending=False)

In [ ]:
# --- 4-Panel Overview ---
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Spend vs Revenue by Channel
x = np.arange(len(channel_summary))
w = 0.35
axes[0,0].bar(x - w/2, channel_summary['Total_Spend']/1000, w, label='Spend (₹K)', color='#e74c3c', alpha=0.85)
axes[0,0].bar(x + w/2, channel_summary['Total_Revenue']/1000, w, label='Revenue (₹K)', color='#2ecc71', alpha=0.85)
axes[0,0].set_xticks(x)
axes[0,0].set_xticklabels(channel_summary['Channel'], rotation=30, ha='right')
axes[0,0].set_title('Spend vs Revenue by Channel', fontweight='bold')
axes[0,0].set_ylabel('Amount (₹ Thousands)')
axes[0,0].legend()

# 2. ROAS by Channel
colors_roas = ['#27ae60' if r >= 3 else '#e67e22' if r >= 2 else '#e74c3c'
               for r in channel_summary['Avg_ROAS']]
bars = axes[0,1].bar(channel_summary['Channel'], channel_summary['Avg_ROAS'], color=colors_roas, edgecolor='white')
axes[0,1].axhline(y=3, color='black', linestyle='--', linewidth=1.5, label='ROAS Target (3x)')
axes[0,1].set_title('Average ROAS by Channel', fontweight='bold')
axes[0,1].set_ylabel('ROAS')
axes[0,1].tick_params(axis='x', rotation=30)
axes[0,1].legend()
for bar, val in zip(bars, channel_summary['Avg_ROAS']):
    axes[0,1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, f'{val:.2f}x',
                   ha='center', fontsize=9)

# 3. CTR vs CVR Scatter
scatter_colors = sns.color_palette('Set2', len(channel_summary))
for i, row in channel_summary.iterrows():
    axes[1,0].scatter(row['Avg_CTR']*100, row['Avg_CVR']*100, s=row['Total_Spend']/500,
                      color=scatter_colors[i], alpha=0.85, edgecolors='white', linewidth=1.5)
    axes[1,0].annotate(row['Channel'], (row['Avg_CTR']*100, row['Avg_CVR']*100),
                       textcoords='offset points', xytext=(6, 4), fontsize=8)
axes[1,0].set_title('CTR vs CVR (Bubble = Spend)', fontweight='bold')
axes[1,0].set_xlabel('Click-Through Rate (%)')
axes[1,0].set_ylabel('Conversion Rate (%)')

# 4. Weekly Revenue Trend by Channel
weekly = df.groupby(['Date','Channel'])['Revenue'].sum().reset_index()
for i, ch in enumerate(df['Channel'].unique()):
    sub = weekly[weekly['Channel'] == ch]
    axes[1,1].plot(sub['Date'], sub['Revenue']/1000, label=ch, linewidth=1.5,
                   color=scatter_colors[i])
axes[1,1].set_title('Weekly Revenue by Channel', fontweight='bold')
axes[1,1].set_xlabel('Date')
axes[1,1].set_ylabel('Revenue (₹ Thousands)')
axes[1,1].legend(fontsize=7)
axes[1,1].tick_params(axis='x', rotation=30)

plt.suptitle('📣 Marketing Campaign Dashboard', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 4️⃣ Conversion Funnel Analysis

In [ ]:
# Aggregate funnel metrics across all channels
total_impressions  = df['Impressions'].sum()
total_clicks       = df['Clicks'].sum()
total_conversions  = df['Conversions'].sum()
total_revenue      = df['Revenue'].sum()

# Estimate Add-to-Cart (between click and conversion)
total_cart = int(total_clicks * 0.18)

funnel_data = {
    'Stage': ['Impressions', 'Clicks', 'Add to Cart', 'Conversions'],
    'Count': [total_impressions, total_clicks, total_cart, total_conversions]
}
funnel_df = pd.DataFrame(funnel_data)
funnel_df['Drop_Rate'] = (1 - funnel_df['Count'] / funnel_df['Count'].shift(1)) * 100

fig = go.Figure(go.Funnel(
    y = funnel_df['Stage'],
    x = funnel_df['Count'],
    textinfo = 'value+percent initial',
    marker = dict(color=['#3498db','#2ecc71','#f39c12','#e74c3c'])
))
fig.update_layout(
    title='🔽 Marketing Conversion Funnel (All Channels)',
    font=dict(size=13),
    height=450
)
fig.show()

print('\n📉 Stage Drop Rates:')
print(funnel_df[['Stage','Count','Drop_Rate']].to_string(index=False))

## 5️⃣ A/B Test Analysis

In [ ]:
np.random.seed(99)

# Simulate A/B test on email subject lines
n_A, n_B = 5000, 5000
conv_A = np.random.binomial(1, 0.062, n_A)  # Control: 6.2% CVR
conv_B = np.random.binomial(1, 0.073, n_B)  # Variant: 7.3% CVR

cr_A = conv_A.mean()
cr_B = conv_B.mean()
uplift = (cr_B - cr_A) / cr_A * 100

# Two-proportion z-test
from statsmodels.stats.proportion import proportions_ztest
count = np.array([conv_A.sum(), conv_B.sum()])
nobs  = np.array([n_A, n_B])
z_stat, p_val = proportions_ztest(count, nobs)

print('='*55)
print('🧪 A/B TEST RESULTS — Email Subject Line Test')
print('='*55)
print(f'  Control (A)   — "Check out our latest deals"')
print(f'  Variant (B)   — "Your exclusive 20% off expires tonight"')
print(f'  Conversion A  : {cr_A:.3%}  ({conv_A.sum()} / {n_A})')
print(f'  Conversion B  : {cr_B:.3%}  ({conv_B.sum()} / {n_B})')
print(f'  Uplift        : +{uplift:.1f}%')
print(f'  Z-Statistic   : {z_stat:.4f}')
print(f'  P-Value       : {p_val:.4f}')
print(f'  Significant?  : {"✅ YES (p < 0.05)" if p_val < 0.05 else "❌ NO (p >= 0.05)"}')
print('='*55)

In [ ]:
# Visualize A/B test
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart
bars = axes[0].bar(['Control (A)', 'Variant (B)'], [cr_A*100, cr_B*100],
                   color=['#3498db','#e74c3c'], edgecolor='white', width=0.5)
axes[0].set_ylabel('Conversion Rate (%)')
axes[0].set_title('A/B Test: Conversion Rate Comparison', fontweight='bold')
for bar, val in zip(bars, [cr_A*100, cr_B*100]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, f'{val:.2f}%',
                 ha='center', fontweight='bold')
axes[0].text(0.5, max(cr_A, cr_B)*100 * 0.6, f'Uplift: +{uplift:.1f}%\np={p_val:.4f}',
             ha='center', transform=axes[0].transAxes,
             fontsize=12, color='green' if p_val < 0.05 else 'red',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# Simulated distribution
n_sim = 10000
sim_A = np.random.binomial(n_A, cr_A, n_sim) / n_A
sim_B = np.random.binomial(n_B, cr_B, n_sim) / n_B
axes[1].hist(sim_A*100, bins=50, alpha=0.6, color='#3498db', label='Control (A)', density=True)
axes[1].hist(sim_B*100, bins=50, alpha=0.6, color='#e74c3c', label='Variant (B)', density=True)
axes[1].set_title('Simulated Sampling Distributions', fontweight='bold')
axes[1].set_xlabel('Conversion Rate (%)')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.suptitle('🧪 A/B Test Statistical Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6️⃣ Marketing Attribution Modeling

In [ ]:
np.random.seed(7)

# Simulate customer journeys (touchpoints before conversion)
channels_list = ['Google Ads', 'Facebook', 'Email', 'Instagram', 'Organic SEO', 'YouTube']

def simulate_journey():
    length = np.random.choice([1, 2, 3, 4, 5], p=[0.3, 0.3, 0.2, 0.15, 0.05])
    return list(np.random.choice(channels_list, size=length, replace=True))

n_conversions = 2000
journeys = [simulate_journey() for _ in range(n_conversions)]
order_values = np.random.uniform(300, 2000, n_conversions)

# Attribution models
first_touch  = {ch: 0 for ch in channels_list}
last_touch   = {ch: 0 for ch in channels_list}
linear_attr  = {ch: 0 for ch in channels_list}

for journey, value in zip(journeys, order_values):
    first_touch[journey[0]]  += value
    last_touch[journey[-1]]  += value
    share = value / len(journey)
    for ch in journey:
        linear_attr[ch] += share

# Build attribution DataFrame
attr_df = pd.DataFrame({
    'Channel': channels_list,
    'First Touch': [first_touch[c] for c in channels_list],
    'Last Touch':  [last_touch[c]  for c in channels_list],
    'Linear':      [linear_attr[c] for c in channels_list]
})

# Normalize to % of total
for col in ['First Touch', 'Last Touch', 'Linear']:
    attr_df[f'{col} %'] = attr_df[col] / attr_df[col].sum() * 100

print('📊 Attribution Model Comparison (% of Revenue Attributed):')
attr_df[['Channel','First Touch %','Last Touch %','Linear %']].round(1)

In [ ]:
# Grouped bar chart: Attribution comparison
x = np.arange(len(channels_list))
w = 0.25

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x - w, attr_df['First Touch %'], w, label='First Touch', color='#3498db', alpha=0.85)
ax.bar(x,     attr_df['Last Touch %'],  w, label='Last Touch',  color='#e74c3c', alpha=0.85)
ax.bar(x + w, attr_df['Linear %'],      w, label='Linear',      color='#2ecc71', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(channels_list, fontsize=11)
ax.set_ylabel('% of Total Revenue Attributed')
ax.set_title('🏷️ Marketing Attribution Model Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 7️⃣ Budget Optimization Recommendations

In [ ]:
# Merge ROAS with attribution to recommend budget allocation
budget_df = channel_summary[['Channel','Total_Spend','Total_Revenue','Avg_ROAS','Avg_ROI']].copy()
budget_df = budget_df.merge(attr_df[['Channel','Linear %']], on='Channel')

total_budget = budget_df['Total_Spend'].sum()

# Score: weighted average of ROAS rank and Linear Attribution rank
budget_df['ROAS_Rank'] = budget_df['Avg_ROAS'].rank(ascending=True)
budget_df['Attr_Rank'] = budget_df['Linear %'].rank(ascending=True)
budget_df['Score']     = (budget_df['ROAS_Rank'] * 0.6 + budget_df['Attr_Rank'] * 0.4)
budget_df['Recommended_Budget_%'] = (budget_df['Score'] / budget_df['Score'].sum() * 100).round(1)
budget_df['Recommended_Spend']    = (budget_df['Recommended_Budget_%'] / 100 * total_budget).round(0)
budget_df['Current_Spend_%']      = (budget_df['Total_Spend'] / total_budget * 100).round(1)
budget_df['Budget_Change']         = budget_df['Recommended_Spend'] - budget_df['Total_Spend']

print('='*70)
print('💡 BUDGET REALLOCATION RECOMMENDATIONS')
print('='*70)
for _, row in budget_df.sort_values('Score', ascending=False).iterrows():
    direction = '⬆️ Increase' if row['Budget_Change'] > 0 else '⬇️ Decrease'
    print(f"  {row['Channel']:15s} | ROAS: {row['Avg_ROAS']:.2f}x | "
          f"Current: {row['Current_Spend_%']:.1f}% → "
          f"Recommended: {row['Recommended_Budget_%']:.1f}% | "
          f"{direction} by ₹{abs(row['Budget_Change']):,.0f}")

# Visual
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(budget_df))
w = 0.35
ax.bar(x - w/2, budget_df['Current_Spend_%'], w, label='Current %', color='#e74c3c', alpha=0.85)
ax.bar(x + w/2, budget_df['Recommended_Budget_%'], w, label='Recommended %', color='#2ecc71', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(budget_df['Channel'], rotation=30, ha='right')
ax.set_ylabel('Budget Allocation (%)')
ax.set_title('💰 Current vs Recommended Budget Allocation', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## ✅ Summary

| Analysis | Key Finding |
|----------|-------------|
| Campaign EDA | Email & Google Ads deliver highest ROAS |
| Funnel Analysis | 82% drop from impressions to clicks — improve ad creative |
| A/B Test | Urgency-based subject line yields +17% uplift (p < 0.05) |
| Attribution | Linear model gives balanced credit vs First/Last touch bias |
| Budget | Reallocate from YouTube → Email and Google Ads for better ROI |

---
**📁 GitHub:** [Your GitHub Link Here]